# env / agent metaparam split — test notebook

Checkpoint §I' A1: verify the new `env_params()` / `agent_params()` /
`grouped_params()` views over the flat metaparameter namespace, and that
the flat API (`params()`, `params_str()`) and the `.evoca` recipe
round-trip are UNCHANGED (bit-for-bit no-op for existing callers).

NOTE: the C library holds a single global lattice, so only ONE `EvoCA`
may be live at a time. Each cell captures reference values, then frees
its sim before the next one is constructed.

Run all cells; the final cell prints PASS/FAIL.

In [1]:
import os, sys
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, 'C', 'evoca.h')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'python'))
from evoca_py import EvoCA, import_run

checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}")

In [2]:
# Init a sim with a non-default value in BOTH groups so the split is
# actually exercised (env: food_inc/gdiff/tax; agent: m_scale/mu_*/tax_*).
# Capture all reference values while it is alive, write a recipe, then
# FREE it (single global lattice -> only one live sim at a time).
sim = EvoCA()
sim.init(32, food_inc=0.013, m_scale=2.0, gdiff=0.4,
         mu_lut=1e-3, mu_egene=2e-3, mu_egenome=4e-3,
         p_dup_egene=0.5, tax=5e-3,
         tax_per_egene=6e-4, tax_lut=7e-4, tax_ring=3e-3,
         restricted_mu=1)

flat  = sim.params()
env   = sim.env_params()
agent = sim.agent_params()
grp   = sim.grouped_params()
pstr  = sim.params_str()

sim.state(lut='gol', egenome='uniform')   # populate _state_params
recipe_path = sim.export_recipe('_env_agent_split_nb')
sim.free()                                # release the global lattice

print('flat  :', flat)
print('env   :', env)
print('agent :', agent)
print('recipe:', os.path.basename(recipe_path))

flat  : {'N': 32, 'food_inc': 0.013, 'm_scale': 2.0, 'gdiff': 0.4, 'mu_lut': 0.001, 'mu_egene': 0.002, 'mu_egenome': 0.004, 'p_dup_egene': 0.5, 'tax': 0.005, 'tax_per_egene': 0.0006, 'tax_lut': 0.0007, 'tax_ring': 0.003, 'restricted_mu': 1}
env   : {'N': 32, 'food_inc': 0.013, 'gdiff': 0.4, 'tax': 0.005}
agent : {'m_scale': 2.0, 'mu_lut': 0.001, 'mu_egene': 0.002, 'mu_egenome': 0.004, 'p_dup_egene': 0.5, 'tax_per_egene': 0.0006, 'tax_lut': 0.0007, 'tax_ring': 0.003, 'restricted_mu': 1}
recipe: 2026-10-15__env_agent_split_nb.evoca


In [3]:
# 1. The two grouped views must PARTITION params() exactly, and be
#    DERIVED from it (single source of truth).
print('Partition / derivation:')
check('env u agent == params() keys', set(env) | set(agent) == set(flat))
check('env n agent == empty', set(env) & set(agent) == set())
check('no metaparam dropped/duplicated',
      len(env) + len(agent) == len(flat))
check('grouped values match flat (single source of truth)',
      all(env[k]   == flat[k] for k in env) and
      all(agent[k] == flat[k] for k in agent))
check('grouped_params() returns {env, agent}',
      grp == {'env': env, 'agent': agent})

# 2. Tax splits by causal owner: baseline tax -> env; complexity taxes -> agent.
print('Tax family split (causal owner):')
check('baseline tax in env',        'tax'           in env)
check('tax_per_egene in agent',     'tax_per_egene' in agent)
check('tax_lut in agent',           'tax_lut'       in agent)
check('tax_ring in agent',          'tax_ring'      in agent)
check('N is an env metaparam',      'N'             in env)

Partition / derivation:
  [PASS] env u agent == params() keys
  [PASS] env n agent == empty
  [PASS] no metaparam dropped/duplicated
  [PASS] grouped values match flat (single source of truth)
  [PASS] grouped_params() returns {env, agent}
Tax family split (causal owner):
  [PASS] baseline tax in env
  [PASS] tax_per_egene in agent
  [PASS] tax_lut in agent
  [PASS] tax_ring in agent
  [PASS] N is an env metaparam


In [4]:
# 3. Flat API unchanged: params() re-inits identically; params_str stable;
#    grouped views reconstruct too. (One live sim, freed before cell 4.)
print('Flat API round-trip:')
sim2 = EvoCA(); sim2.init(**flat)
check('params() reconstructs an identical sim', sim2.params() == flat)
check('params_str() stable across re-init',     sim2.params_str() == pstr)
check('env/agent views reconstruct',
      sim2.env_params() == env and sim2.agent_params() == agent)
sim2.free()

Flat API round-trip:
  [PASS] params() reconstructs an identical sim
  [PASS] params_str() stable across re-init
  [PASS] env/agent views reconstruct


In [5]:
# 4. .evoca recipe export/import still round-trips unchanged. Each import
#    is freed before the next (single global lattice).
print('.evoca recipe round-trip:')
try:
    for kind in ('init', 'final'):
        s, _kw = import_run(recipe_path, recipe=kind)
        try:
            same = all(s.params()[k] == flat[k] for k in flat if k != 'N')
            check(f'metaparams_{kind} round-trip', same)
            check(f'env/agent views match after {kind} import',
                  s.env_params() == env and s.agent_params() == agent)
        finally:
            s.free()
finally:
    if os.path.exists(recipe_path):
        os.remove(recipe_path)

.evoca recipe round-trip:
  [PASS] metaparams_init round-trip
  [PASS] env/agent views match after init import
  [PASS] metaparams_final round-trip
  [PASS] env/agent views match after final import


In [6]:
n_fail = sum(1 for _, ok in checks if not ok)
print('\n' + '=' * 40)
print(f'{len(checks)} checks, {n_fail} failed')
print('OVERALL:', 'PASS' if n_fail == 0 else 'FAIL')
assert n_fail == 0, 'env/agent split notebook FAILED'


17 checks, 0 failed
OVERALL: PASS
